# Lecture 2 — SOLUTIONS: Signal and noise — from single genes to signatures

**Guiding idea.** Bulk RNA-seq, qPCR and in situ hybridisation average over millions of cells, so every gene
is read through many molecules and its relative sampling noise (≈ 1/√λ, with λ the expected count) is
negligible. That is why **one marker gene** (*PAX6*, *EOMES*, *TBR1*…) used to be enough, and why the
bulk benchmarking of organoids against BrainSpan (PCA, correlation, WGCNA) works.

Single-cell and Visium HD measure **one cell or one bin**: a few hundred to a few thousand molecules spread
over ~20,000 genes. For most genes λ < 1, so **zero is the most likely reading even when the gene is
expressed**: P(0) = e^(−λ). At this resolution a single gene is no longer a reliable readout, and information
must be pooled:

- **across genes**: signatures and multi-gene scores, modules (WGCNA), PCA — averaging *k* genes improves
  signal-to-noise roughly as √k;
- **across space**: larger bins, smoothing over neighbouring bins;
- **across cells**: clusters and pseudobulk.

A note on words: in droplet UMI data most zeros are *not* a technical "dropout" artefact but plain Poisson /
negative binomial sampling at low depth (Svensson 2020). The problem is that counts are few, not that the
technology is broken. Exercise 3 checks this on real data.

Same way of working as in Lecture 1: read the question, complete the **decomposition tree**, write
**one function per leaf** and a root function that calls them.

**Datasets** (in `Datasets/`):
- `e18_mouse_brain_*` — 1,301 cells from embryonic day 18 mouse cortex, hippocampus and ventricular zone
  (10x Genomics public dataset *1k Brain Cells from an E18 Mouse*, Chromium v3). Sparse count matrix
  cells × genes (`scipy.sparse`) + gene names.
- `visiumhd_mouse_brain_*` — crops of a 10x Genomics Visium HD section of the adult mouse brain
  (public dataset *Visium HD Mouse Brain, fresh frozen*), at 2, 8 and 16 µm bins, for a panel of marker genes.
  For each resolution: a sparse count matrix bins × genes, and a table of bins with grid position and total UMIs
  (over **all** genes, not only the panel).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, ndimage, sparse
from scipy.cluster.vq import kmeans2

rng = np.random.default_rng(2026)

sc_counts = sparse.load_npz("../Datasets/e18_mouse_brain_counts.npz").tocsr()   # cells × genes
sc_genes = pd.read_csv("../Datasets/e18_mouse_brain_genes.csv")["gene"].to_numpy()
print("single-cell:", sc_counts.shape)

## Exercise 1 — How many molecules do you need to see a gene? (simulation)

**Question:** take the same expression profile (the fraction of mRNA molecules that comes from each gene)
and read it at the depth of a bulk library, of a good single cell, of a poor single cell and of a Visium HD
bin. What happens to each gene?

The expected count of gene *g* is λ_g = depth × fraction_g, and the observed count is Poisson(λ_g).

```
From bulk to a single bin
├── Profile          → make_profile(n_genes): log-normal fractions that sum to 1
├── Read it          → read_out(profile, depth, n_obs): Poisson counts, n_obs observations
├── Detection        → detection_rate(counts): fraction of observations with count > 0, per gene
├── Precision        → cv(counts): standard deviation / mean, per gene
└── Plot             → detection and CV vs λ, one curve per depth, with the theory 1 − e^(−λ) and 1/√λ
```

**To do:** use depths 20,000,000 (bulk), 10,000 (good cell), 2,000 (poor cell), 150 (8 µm Visium HD bin)
and 10 (2 µm bin): the last two are the median total UMIs per bin of the Visium HD section of Exercise 4. For each depth, report how many genes have λ ≥ 10 (≈ a CV below 30%).

*Discussion question:* at which depth would you still trust "gene X is not expressed in this cell"?

In [ ]:
DEPTHS = {"bulk": 20_000_000, "good cell": 10_000, "poor cell": 2_000, "8 µm bin": 150, "2 µm bin": 10}

def make_profile(n_genes=15_000, rng=rng):
    w = rng.lognormal(mean=0, sigma=2, size=n_genes)
    return w / w.sum()

def read_out(profile, depth, n_obs=300, rng=rng):
    return rng.poisson(profile * depth, size=(n_obs, len(profile)))

def detection_rate(counts):
    return (counts > 0).mean(axis=0)

def cv(counts):
    mean = counts.mean(axis=0)
    with np.errstate(divide="ignore", invalid="ignore"):
        return counts.std(axis=0) / mean

def summarise_depths(profile):
    rows = []
    for name, depth in DEPTHS.items():
        lam = profile * depth
        rows.append({"readout": name, "depth": depth,
                     "median λ": np.median(lam),
                     "genes with λ ≥ 1": int((lam >= 1).sum()),
                     "genes with λ ≥ 10": int((lam >= 10).sum())})
    return pd.DataFrame(rows)

def plot_detection_and_cv(profile, n_genes_plot=1500):
    genes = rng.choice(len(profile), n_genes_plot, replace=False)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    lam_grid = np.logspace(-3, 3, 200)
    for name, depth in list(DEPTHS.items())[1:]:          # bulk is off the scale
        counts = read_out(profile[genes], depth)
        lam = profile[genes] * depth
        ax1.scatter(lam, detection_rate(counts), s=3, label=name)
        ax2.scatter(lam, cv(counts), s=3, label=name)
    ax1.plot(lam_grid, 1 - np.exp(-lam_grid), "k--", lw=1, label="1 − e^(−λ)")
    ax2.plot(lam_grid, 1 / np.sqrt(lam_grid), "k--", lw=1, label="1/√λ")
    for ax in (ax1, ax2):
        ax.set_xscale("log")
        ax.set_xlabel("expected count λ")
        ax.set_xlim(1e-3, 1e3)
    ax1.set_ylabel("fraction of observations with count > 0")
    ax2.set_ylabel("coefficient of variation")
    ax2.set_yscale("log")
    ax1.legend(markerscale=4)
    plt.tight_layout()
    plt.show()

profile = make_profile()
display(summarise_depths(profile))
plot_detection_and_cv(profile)

**Comment.** In bulk, essentially every gene has λ ≫ 10. In a good cell only a few hundred genes are
read precisely, and in a Visium HD bin almost none: for nearly every gene the most likely reading is 0,
and the CV of a single gene is ≥ 1. Nothing is "broken": it is just the arithmetic of few molecules.

## Exercise 2 — One marker or a signature?

### 2a — Simulation

**Question:** two populations (say radial glia and neurons) differ by a factor 3 in 50 marker genes.
How well can we tell them apart using 1, 2, 5, 10, 20 or 50 of the markers, at different depths?

We measure separability with the **AUC**: the probability that a random cell of population B has a higher
score than a random cell of population A (0.5 = coin toss, 1 = perfect). It is the Mann–Whitney U statistic
divided by n_A × n_B.

```
Marker vs signature
├── Simulate         → simulate_two_populations(depth, n_markers=50, fold=3): counts for A and B
├── Normalise        → log1p(counts / total × 10,000)
├── Score            → signature_score(norm, markers): mean of the normalised marker genes
├── Separability     → auc(score_a, score_b)
└── Grid             → AUC for k ∈ {1, 2, 5, 10, 20, 50} × depth ∈ {200, 1,000, 5,000, 20,000}
```

*Discussion question:* the gain from adding markers is large at low depth and small at high depth. Why?
What would happen if half of your "markers" were not really markers?

In [ ]:
def simulate_two_populations(depth, n_markers=50, n_background=2000, fold=3.0, n_cells=300, rng=rng):
    background = rng.lognormal(0, 1.5, n_background)
    background = background / background.sum() * (1 - n_markers * 2e-4)
    base = np.concatenate([np.full(n_markers, 2e-4), background])   # each marker: 0.02% of the molecules
    profile_b = base.copy()
    profile_b[:n_markers] *= fold
    profile_b = profile_b / profile_b.sum()
    depth_a = depth * rng.lognormal(0, 0.3, n_cells)        # cells differ in depth too
    depth_b = depth * rng.lognormal(0, 0.3, n_cells)
    a = rng.poisson(np.outer(depth_a, base))
    b = rng.poisson(np.outer(depth_b, profile_b))
    return a, b

def normalise(counts, target=10_000):
    totals = counts.sum(axis=1, keepdims=True)
    return np.log1p(counts / np.maximum(totals, 1) * target)

def signature_score(norm, markers):
    return norm[:, markers].mean(axis=1)

def auc(score_a, score_b):
    u = stats.mannwhitneyu(score_b, score_a).statistic
    return u / (len(score_a) * len(score_b))

def auc_grid(ks=(1, 2, 5, 10, 20, 50), depths=(200, 1_000, 5_000, 20_000)):
    table = {}
    for depth in depths:
        a, b = simulate_two_populations(depth)
        na, nb = normalise(a), normalise(b)
        table[depth] = {k: auc(signature_score(na, np.arange(k)), signature_score(nb, np.arange(k))) for k in ks}
    return pd.DataFrame(table).rename_axis(index="markers used", columns="depth (UMIs per cell)")

grid = auc_grid()
display(grid.round(3))
grid.plot(marker="o", logx=True, figsize=(5, 3.5), ylabel="AUC", title="AUC vs number of markers")
plt.show()

### 2b — Real data: does a progenitor have *Pax6*?

In the E18 mouse brain cells we score each cell with a **radial glia / progenitor signature** and look
at how often the classic single marker *Pax6* is zero in cells that the signature calls progenitors.

```
Single marker vs signature on real cells
├── Normalise        → log1p(counts / total × 10,000)
├── Signature score  → mean of the progenitor genes (not including the single markers we test)
├── Define groups    → top 20% of the score = "progenitor-like", bottom 50% = "not progenitor"
└── Compare          → fraction of Pax6 = 0 in each group; same for the neuronal marker Neurod6 and a neuronal signature
```

*Discussion question:* if you sorted cells by "*Pax6* > 0", what fraction of the progenitor-like cells
would you lose? Is the signature "true"? How would you check it does not just reflect depth?

In [ ]:
# the single markers we test (Pax6, Sox2, Neurod6, Tbr1) are NOT in the signatures, to avoid circularity
PROGENITOR = ["Hes1", "Hes5", "Vim", "Nes", "Fabp7", "Slc1a3", "Mki67", "Top2a", "Ccnd2", "Notch1", "Fgfr3", "Sox9"]
NEURON = ["Neurod2", "Bcl11b", "Satb2", "Stmn2", "Tubb3", "Mef2c", "Gap43", "Syt1", "Nrxn1", "Dcx"]

def gene_index(names):
    lookup = {g: i for i, g in enumerate(sc_genes)}
    missing = [g for g in names if g not in lookup]
    if missing:
        print("not in the data:", missing)
    return [lookup[g] for g in names if g in lookup]

def normalise_sparse(counts, target=10_000):
    totals = np.asarray(counts.sum(axis=1)).ravel()
    return counts.multiply(target / totals[:, None]).log1p().tocsc()

def score(norm, genes):
    return np.asarray(norm[:, gene_index(genes)].mean(axis=1)).ravel()

def zero_fraction_by_group(counts, gene, score_values):
    raw = counts[:, gene_index([gene])].toarray().ravel()
    high = score_values >= np.quantile(score_values, 0.8)
    low = score_values <= np.quantile(score_values, 0.5)
    return {"gene": gene, "zeros in top 20% of signature": (raw[high] == 0).mean(),
            "zeros in bottom 50% of signature": (raw[low] == 0).mean()}

sc_norm = normalise_sparse(sc_counts)
progenitor_score = score(sc_norm, PROGENITOR)
neuron_score = score(sc_norm, NEURON)

display(pd.DataFrame([
    zero_fraction_by_group(sc_counts, "Pax6", progenitor_score),
    zero_fraction_by_group(sc_counts, "Sox2", progenitor_score),
    zero_fraction_by_group(sc_counts, "Neurod6", neuron_score),
    zero_fraction_by_group(sc_counts, "Tbr1", neuron_score),
]).round(2))

plt.figure(figsize=(5, 4))
pax6 = sc_counts[:, gene_index(["Pax6"])].toarray().ravel()
plt.scatter(progenitor_score, neuron_score, c=pax6 > 0, cmap="coolwarm", s=6)
plt.xlabel("progenitor signature")
plt.ylabel("neuronal signature")
plt.title("red = Pax6 detected, blue = Pax6 = 0")
plt.show()

## Exercise 3 — Zeros in real data: dropout or sampling?

**Question:** are there *more* zeros in the E18 data than expected from simple sampling? If each gene *g*
has a fraction μ_g of the molecules and cell *i* has total *s_i*, the Poisson model predicts

$$P(\text{zero}_g) = \frac{1}{N}\sum_i e^{-\mu_g s_i}$$

and a negative binomial with overdispersion φ (biological variability between cells) predicts
$\frac{1}{N}\sum_i (1+\varphi\,\mu_g s_i)^{-1/\varphi}$.

```
Zero inflation check
├── Per gene         → mean count, observed fraction of zeros
├── Poisson          → expected_zeros_poisson(mu, totals)
├── Neg. binomial    → expected_zeros_nb(mu, totals, phi) for phi in {0.1, 0.5}
└── Plot             → observed vs expected zero fraction against mean count (log x)
```

*Discussion question:* the cells are a mixture of different cell types. Why does this alone produce
"extra" zeros compared to the Poisson line? So, is a gene that is zero in a cell "dropped out"?

In [ ]:
def gene_stats(counts):
    totals = np.asarray(counts.sum(axis=1)).ravel()
    mean = np.asarray(counts.mean(axis=0)).ravel()
    zeros = 1 - np.asarray((counts > 0).mean(axis=0)).ravel()
    mu = np.asarray(counts.sum(axis=0)).ravel() / totals.sum()
    return pd.DataFrame({"mean": mean, "zeros": zeros, "mu": mu}), totals

def expected_zeros_poisson(mu, totals):
    return np.exp(-np.outer(mu, totals)).mean(axis=1)

def expected_zeros_nb(mu, totals, phi):
    return ((1 + phi * np.outer(mu, totals)) ** (-1 / phi)).mean(axis=1)

def plot_zeros(df):
    order = np.argsort(df["mean"].to_numpy())
    plt.figure(figsize=(6, 4))
    plt.scatter(df["mean"], df["zeros"], s=2, alpha=0.3, color="grey", label="observed genes")
    for col, style in [("poisson", "k-"), ("nb_0.1", "C0-"), ("nb_0.5", "C3-")]:
        plt.plot(df["mean"].to_numpy()[order], df[col].to_numpy()[order], style, lw=1.5, label=col)
    plt.xscale("log")
    plt.xlabel("mean count per cell")
    plt.ylabel("fraction of cells with zero")
    plt.legend()
    plt.show()

zero_df, sc_totals = gene_stats(sc_counts)
zero_df["poisson"] = expected_zeros_poisson(zero_df["mu"].to_numpy(), sc_totals)
for phi in (0.1, 0.5):
    zero_df[f"nb_{phi}"] = expected_zeros_nb(zero_df["mu"].to_numpy(), sc_totals, phi)
plot_zeros(zero_df)
print("median excess of zeros over Poisson:", round((zero_df["zeros"] - zero_df["poisson"]).median(), 3))

**Comment.** Most genes sit close to the Poisson/NB curves: their zeros are what you expect from
reading few molecules. The excess for some genes comes from biology (they are expressed only in some cell
types), not from a technical failure. Either way, a zero in one cell is weak evidence of absence.

## Exercise 4 — Visium HD: resolution against signal

The Visium HD file `visiumhd_mouse_brain_cortex_2um.npz` is a 1.7 × 1.7 mm crop of an adult mouse brain
section at the native **2 µm** resolution: from the striatum (left) through the white matter to the cortex and
the pia (right). It contains 67 marker genes and, for every bin, the total UMIs over **all** ~19,000 genes.
`visiumhd_mouse_brain_gene_means.csv` gives, for every gene, its mean count per bin over the whole section
at 2, 8 and 16 µm.

The 8 and 16 µm bins of the 10x pipeline are simply sums of 4×4 and 8×8 bins of 2 µm: we can rebuild them
ourselves.

### 4a — How many molecules per bin?

```
Counts per bin at each resolution
├── Load            → load_visium(path): sparse counts (bins × genes), bins table (row, col, total_counts), genes
├── Aggregate       → aggregate(counts, bins, size_um): sum b×b neighbouring 2 µm bins (b = size / 2)
├── Summary         → summarise_resolution(...): bins, median total UMIs, median and % zero for a few genes
└── Whole section   → from the gene-means table: genes with mean ≥ 1 and ≥ 10 per bin, per resolution
```

Compare with Exercise 1: which line of that table does a Visium HD bin correspond to?

In [ ]:
def load_visium(path):
    z = np.load(path)
    counts = sparse.csr_matrix((z["data"], z["indices"], z["indptr"]), shape=tuple(z["shape"]))
    bins = pd.DataFrame({"row": z["row"].astype(int), "col": z["col"].astype(int),
                         "total_counts": z["total_counts"].astype(float)})
    return counts, bins, list(z["genes"])

def aggregate(counts, bins, size_um):
    b = size_um // 2
    key = (bins["row"] // b).to_numpy() * 100_000 + (bins["col"] // b).to_numpy()
    unique, inverse = np.unique(key, return_inverse=True)
    assign = sparse.csr_matrix((np.ones(len(key)), (inverse, np.arange(len(key)))), shape=(len(unique), len(key)))
    new_bins = pd.DataFrame({"row": unique // 100_000, "col": unique % 100_000,
                             "total_counts": np.bincount(inverse, weights=bins["total_counts"])})
    return (assign @ counts).tocsr(), new_bins

def summarise_resolution(counts, bins, genes, size_um, example_genes=("Slc17a7", "Mbp", "Rorb")):
    agg, agg_bins = aggregate(counts, bins, size_um)
    row = {"bin size (µm)": size_um, "bins": len(agg_bins),
           "median total UMIs (all genes)": np.median(agg_bins["total_counts"])}
    for g in example_genes:
        x = agg[:, genes.index(g)].toarray().ravel()
        row[f"{g}: mean"] = x.mean()
        row[f"{g}: % zero"] = 100 * np.mean(x == 0)
    return row

vh_counts, vh_bins, vh_genes = load_visium("../Datasets/visiumhd_mouse_brain_cortex_2um.npz")
print("2 µm crop:", vh_counts.shape)
display(pd.DataFrame([summarise_resolution(vh_counts, vh_bins, vh_genes, s) for s in (2, 4, 8, 16, 32)]).round(2))

gene_means = pd.read_csv("../Datasets/visiumhd_mouse_brain_gene_means.csv")
display(pd.DataFrame({
    col.replace("mean_per_bin_", ""): {"genes with mean ≥ 1": (gene_means[col] >= 1).sum(),
                                       "genes with mean ≥ 10": (gene_means[col] >= 10).sum()}
    for col in gene_means.columns[1:]}).T)

### 4b — One gene or a signature, in space

The layer-4/5 marker *Rorb* should draw a band parallel to the pia. Compare it with a **layer 4/5
signature** of 5 genes, at 8 and 16 µm, and with the single gene smoothed over neighbouring bins.

```
Maps of one gene vs a signature
├── Score per bin      → bin_score(counts, bins, genes): log1p(1000 × sum of the genes / total UMIs)
├── To an image        → to_image(values, bins): 2D array, NaN where there is no tissue
├── Smooth             → Gaussian filter on the single-gene image (σ = 2 bins)
└── Grid of maps       → rows: 8 µm, 16 µm; columns: Rorb, Rorb smoothed, L4/5 signature
```

### 4c — How reproducible is a bin?

We cannot measure the same bin twice, but with **count splitting** (each molecule goes to half A or half B
with probability 0.5, see also Exercise 6b) we get two independent half-measurements of every bin.
The correlation between the two halves across bins says how much of what we see is signal.

```
Split-half reliability
├── Split          → split_counts(counts): Binomial(counts, 0.5) and the rest (totals split the same way)
├── Score halves   → bin_score on each half
├── Correlate      → Pearson r between half A and half B across bins
└── Table          → r for single genes and for signatures, at 2, 4, 8, 16 and 32 µm
```

*Discussion question:* how large must a bin be before *Rorb* alone is reproducible? Is that still
"single-cell resolution"? Does the signature need the same bin size?

In [ ]:
SIGNATURES = {
    "L2/3 IT":          ["Cux2", "Lamp5", "Otof", "Calb1", "Stard8", "Rasgrf2"],
    "L4/5 IT":          ["Rorb", "Rspo1", "Whrn", "Scnn1a", "Kcnh5"],
    "L5 ET":            ["Fezf2", "Bcl11b", "Pou3f1", "Parm1"],
    "L6":               ["Foxp2", "Syt6", "Tle4", "Crym", "Nxph4", "Ccn2"],
    "Oligodendrocytes": ["Mbp", "Plp1", "Mobp", "Mog", "Mag", "Cnp", "Cldn11"],
    "Striatal MSN":     ["Ppp1r1b", "Penk", "Drd1", "Drd2", "Tac1", "Adora2a", "Rgs9", "Gpr88"],
    "Astrocytes":       ["Gfap", "Aqp4", "Aldh1l1", "Slc1a3", "Gja1", "Slc1a2"],
    "Inhibitory":       ["Gad1", "Gad2", "Slc32a1", "Pvalb", "Sst", "Vip"],
    "Microglia":        ["P2ry12", "Csf1r", "Hexb", "C1qa", "Tmem119"],
    "Endothelial":      ["Cldn5", "Flt1", "Pecam1"],
}

def bin_score(counts, bins, genes):
    idx = [vh_genes.index(g) for g in genes]
    signal = np.asarray(counts[:, idx].sum(axis=1)).ravel()
    return np.log1p(1000 * signal / np.maximum(bins["total_counts"].to_numpy(), 1))

def to_image(values, bins):
    image = np.full((bins["row"].max() + 1, bins["col"].max() + 1), np.nan)
    image[bins["row"], bins["col"]] = values
    return image

def smooth_image(image, sigma=2):
    filled = np.nan_to_num(image)
    mask = ndimage.gaussian_filter((~np.isnan(image)).astype(float), sigma)
    out = ndimage.gaussian_filter(filled, sigma) / np.maximum(mask, 1e-9)
    out[np.isnan(image)] = np.nan
    return out

def plot_maps(sizes=(8, 16), gene="Rorb", signature="L4/5 IT"):
    fig, axes = plt.subplots(len(sizes), 3, figsize=(12, 4 * len(sizes)))
    for row, size in zip(axes, sizes):
        agg, agg_bins = aggregate(vh_counts, vh_bins, size)
        single = to_image(bin_score(agg, agg_bins, [gene]), agg_bins)
        panels = [(single, f"{gene}, {size} µm"),
                  (smooth_image(single), f"{gene} smoothed, {size} µm"),
                  (to_image(bin_score(agg, agg_bins, SIGNATURES[signature]), agg_bins),
                   f"{signature} signature, {size} µm")]
        for ax, (img, title) in zip(row, panels):
            ax.imshow(img, cmap="magma", vmax=np.nanquantile(img, 0.99))
            ax.set_title(title)
            ax.axis("off")
    plt.tight_layout()
    plt.show()

plot_maps()

In [ ]:
def split_counts(counts, bins, rng=rng):
    half = counts.copy()
    half.data = rng.binomial(counts.data.astype(np.int64), 0.5)
    other = counts - half
    total_half = rng.binomial(bins["total_counts"].to_numpy().astype(np.int64), 0.5)
    bins_a = bins.assign(total_counts=total_half)
    bins_b = bins.assign(total_counts=bins["total_counts"] - total_half)
    return (half, bins_a), (other, bins_b)

def split_half_reliability(counts, bins, genes):
    (ca, ba), (cb, bb) = split_counts(counts, bins)
    return np.corrcoef(bin_score(ca, ba, genes), bin_score(cb, bb, genes))[0, 1]

def reliability_table(features, sizes=(2, 4, 8, 16, 32)):
    table = {}
    for size in sizes:
        agg, agg_bins = aggregate(vh_counts, vh_bins, size)
        table[f"{size} µm"] = {name: split_half_reliability(agg, agg_bins, genes) for name, genes in features.items()}
    return pd.DataFrame(table)

features = {"Rorb (1 gene)": ["Rorb"], "L4/5 IT (5 genes)": SIGNATURES["L4/5 IT"],
            "Mbp (1 gene)": ["Mbp"], "Oligodendrocytes (7 genes)": SIGNATURES["Oligodendrocytes"],
            "Ppp1r1b (1 gene)": ["Ppp1r1b"], "Striatal MSN (8 genes)": SIGNATURES["Striatal MSN"]}
reliability_table(features).round(2)

**Comment.** At 2–8 µm, the two halves of the same bin barely agree for a single gene: most of what
the map shows is sampling noise. Signatures reach the same reliability at a much smaller bin size, and so do
genes that are very highly expressed in a large structure (*Mbp* in the white matter). Signal must be paid for
with either **space** (bigger bins, smoothing) or **genes** (signatures).

## Exercise 5 — Populations come from single-cell data

The cell types of a spatial section are not discovered in the spatial data: they come from **single-cell
atlases** (for the mouse brain: Zeisel et al. 2018, Tasic et al. 2018, Yao et al. 2021), and the spatial data are
annotated by transferring them. The lists in `SIGNATURES` are marker genes of the atlas cell types.
Real tools (RCTD, cell2location, Tangram) use the full single-cell profiles; here we use the simplest version:
signature scores.

```
Label transfer by signature scores
├── Score matrix      → score_matrix(counts, bins): one column per signature (bin_score), z-scored across bins
├── Label             → the signature with the highest z-score
├── Confidence        → margin = best z − second best z; "ambiguous" if margin < 0.5
├── Maps              → label map at 8 µm, 8 µm with scores smoothed over neighbours, 16 µm and 32 µm
└── Summary           → % ambiguous bins and label composition per resolution
```

*Discussion question:* small bins are ambiguous because they are noisy; big bins are ambiguous because
they contain several cells. Is there a "right" bin size? Why do the spatial data still need the single-cell
reference, and what happens to a cell type that is missing from it (for example, a cell type that exists only
in organoids)?

In [ ]:
def score_matrix(counts, bins):
    scores = pd.DataFrame({name: bin_score(counts, bins, genes) for name, genes in SIGNATURES.items()})
    return (scores - scores.mean()) / scores.std()

def label_bins(z, min_margin=0.5):
    ordered = np.sort(z.to_numpy(), axis=1)
    margin = ordered[:, -1] - ordered[:, -2]
    labels = z.idxmax(axis=1).where(margin >= min_margin, "ambiguous")
    return labels, margin

def smooth_scores(z, bins, sigma):
    out = {}
    for name in z.columns:
        img = smooth_image(to_image(z[name].to_numpy(), bins), sigma)
        out[name] = img[bins["row"], bins["col"]]
    return pd.DataFrame(out)

def plot_labels(panels=((8, 0), (8, 3), (16, 0), (32, 0))):
    names = list(SIGNATURES) + ["ambiguous"]
    colours = plt.get_cmap("tab10").colors + ((0.85, 0.85, 0.85),)
    cmap = plt.matplotlib.colors.ListedColormap(colours[:len(names)])
    fig, axes = plt.subplots(1, len(panels), figsize=(4.5 * len(panels), 5))
    summary = {}
    for ax, (size, sigma) in zip(axes, panels):
        agg, agg_bins = aggregate(vh_counts, vh_bins, size)
        z = score_matrix(agg, agg_bins)
        if sigma > 0:
            z = smooth_scores(z, agg_bins, sigma)
            z = (z - z.mean()) / z.std()
        labels, margin = label_bins(z)
        codes = labels.map({n: i for i, n in enumerate(names)}).to_numpy(dtype=float)
        ax.imshow(to_image(codes, agg_bins), cmap=cmap, vmin=-0.5, vmax=len(names) - 0.5, interpolation="nearest")
        title = f"{size} µm" + (f", scores smoothed (σ = {sigma} bins)" if sigma else "")
        ax.set_title(title, fontsize=10)
        ax.axis("off")
        summary[title] = labels.value_counts(normalize=True).reindex(names).fillna(0) * 100
    handles = [plt.matplotlib.patches.Patch(color=colours[i], label=n) for i, n in enumerate(names)]
    fig.legend(handles=handles, loc="lower center", ncol=6)
    plt.tight_layout(rect=(0, 0.1, 1, 1))
    plt.show()
    return pd.DataFrame(summary).round(1).rename_axis("% of bins")

plot_labels()

**Comment.** Raw 8 µm bins give confetti: about a third of the bins are ambiguous and the rest is mostly
noise. Smoothing the scores over neighbouring bins recovers the anatomy (striatum, white matter, then layers
6 → 5 → 4/5 → 2/3 and the glia at the pia), but it also *creates* blobs (the isolated microglia, endothelial or
L5 ET patches) from noise: smoothing trades resolution for signal and can invent structure. Larger bins
average real mixtures of cells, so a single label per bin becomes less meaningful.

## Exercise 6 — Analysis traps

| Part | Data | Trap |
|---|---|---|
| 6a | E18 cells | sequencing depth as a confounder, and what normalisation does (not) fix |
| 6b | simulation | double dipping: clustering and testing on the same data |
| 6c | simulation | pseudoreplication: cells are not biological replicates |
| 6d | Visium HD | spillover: a marker where its cells are not |

### 6a — Deeper cells "express more" of everything

**Question:** take the E18 cells, pick a random half and **thin** their counts to 30% (each molecule kept with
probability 0.3). Same cells, same biology: only the sequencing depth differs. Then test the 2,000 most
expressed genes between the two halves, with three versions of the data:

```
Depth as a confounder
├── Thin          → thin_cells(counts, fraction=0.3): binomial thinning of a random half of the cells
├── Versions      → raw counts · counts / total (proportions) · log1p(counts / total × 10,000)
├── Test          → t-test per gene, fraction of genes with p < 0.01 (should be ≈ 1%)
└── Real data     → the same test on proportions, splitting the real cells by depth (above / below the median)
```

*Discussion question:* why does the log1p version still "see" depth, although it is divided by the total?
(hint: E[log(X)] ≠ log(E[X]) when X is often 0 — Lun 2018.) And in the real split, why are so many genes still
different after normalisation: what else differs between deep and shallow cells?

In [ ]:
def thin_cells(counts, fraction=0.3, rng=rng):
    thinned_group = rng.random(counts.shape[0]) < 0.5
    thinned = counts.tocsr(copy=True).astype(np.int64)
    for i in np.where(thinned_group)[0]:
        start, end = thinned.indptr[i], thinned.indptr[i + 1]
        thinned.data[start:end] = rng.binomial(thinned.data[start:end], fraction)
    return thinned, thinned_group

def versions(counts, n_genes=2000):
    totals = np.asarray(counts.sum(axis=1)).ravel()[:, None]
    top = np.argsort(np.asarray(counts.sum(axis=0)).ravel())[::-1][:n_genes]
    raw = counts[:, top].toarray().astype(float)
    return {"raw counts": raw,
            "counts / total": raw / totals,
            "log1p(counts / total × 10⁴)": np.log1p(raw / totals * 10_000)}

def fraction_significant_genes(matrix, group, alpha=0.01):
    return np.mean(stats.ttest_ind(matrix[group], matrix[~group]).pvalue < alpha)

def depth_confounding(counts):
    thinned, group = thin_cells(counts)
    result = {f"thinned vs not, {name}": fraction_significant_genes(m, group)
              for name, m in versions(thinned).items()}
    totals = np.asarray(counts.sum(axis=1)).ravel()
    deep = totals > np.median(totals)
    result["real deep vs shallow cells, counts / total"] = fraction_significant_genes(
        versions(counts)["counts / total"], deep)
    return pd.Series(result, name="fraction of genes with p < 0.01").round(3)

depth_confounding(sc_counts)

### 6b — Double dipping

**Question:** simulate cells that are **all the same** (pure Poisson noise, no populations), cluster them
into 2 clusters, then test every gene between the two clusters. How many genes are "differentially
expressed"?

The fix shown here is **count splitting** (Neufeld et al. 2024): for Poisson data, `train = Binomial(X, 0.5)`
and `test = X − train` are independent. Cluster on `train`, test on `test`.

```
Double dipping
├── Simulate      → 500 identical cells × 200 genes, Poisson
├── Cluster       → log1p, PCA (5 components), k-means with 2 clusters
├── Test          → t-test of every gene between clusters, on the same data
├── Split         → count splitting: cluster on train, test on test
└── Compare       → fraction of genes with p < 0.05 (should be ≈ 5%)
```

*Discussion question:* in a real analysis, the clusters you test are the ones you found because they
differed. What does a p-value mean then? When is it still useful to look at "markers" of clusters?

In [ ]:
def simulate_identical_cells(n_cells=500, n_genes=200, rng=rng):
    means = rng.lognormal(0, 1, n_genes)
    return rng.poisson(means, size=(n_cells, n_genes))

def cluster(counts, n_pcs=5, seed=0):
    x = np.log1p(counts)
    x = x - x.mean(axis=0)
    _, _, vt = np.linalg.svd(x, full_matrices=False)
    _, labels = kmeans2(x @ vt[:n_pcs].T, 2, seed=seed, minit="++")
    return labels

def fraction_significant(counts, labels, alpha=0.05):
    x = np.log1p(counts)
    p = stats.ttest_ind(x[labels == 0], x[labels == 1]).pvalue
    return np.mean(p < alpha)

def count_split(counts, rng=rng):
    train = rng.binomial(counts, 0.5)
    return train, counts - train

def double_dipping():
    X = simulate_identical_cells()
    train, test = count_split(X)
    return pd.Series({
        "same data for clustering and testing": fraction_significant(X, cluster(X)),
        "count splitting (cluster train, test test)": fraction_significant(test, cluster(train)),
    }, name="fraction of genes with p < 0.05")

double_dipping()

### 6c — Pseudoreplication: cells are not replicates

**Question:** 3 control and 3 treated organoids, 300 cells each. The treatment does **nothing**, but every
organoid has its own expression level of the gene (organoid-to-organoid variability). Test with
n = cells, then with n = organoids (**pseudobulk**: sum the counts of each organoid).

```
Pseudoreplication
├── Simulate       → simulate_experiment(): organoid level ~ log-normal(sd 0.3), cells Poisson around it
├── Cell-level     → Mann–Whitney on 900 vs 900 cells
├── Pseudobulk     → sum per organoid, divide by number of cells, t-test on log values, 3 vs 3
└── Repeat         → false positive rate over 300 experiments
```

*Discussion question:* in organoid studies, what is the replicate: the organoid, the differentiation
batch, or the iPSC line? What does Squair et al. (2021) recommend?

In [ ]:
def simulate_experiment(n_organoids=3, n_cells=300, mean=2.0, organoid_sd=0.3, rng=rng):
    data = []
    for condition in ["control", "treated"]:
        for o in range(n_organoids):
            level = mean * rng.lognormal(0, organoid_sd)
            counts = rng.poisson(level * rng.lognormal(0, 0.2, n_cells))
            data.append(pd.DataFrame({"condition": condition, "organoid": f"{condition}_{o}", "count": counts}))
    return pd.concat(data, ignore_index=True)

def cell_level_test(df):
    return stats.mannwhitneyu(df.loc[df.condition == "control", "count"],
                              df.loc[df.condition == "treated", "count"]).pvalue

def pseudobulk(df):
    return df.groupby(["condition", "organoid"])["count"].mean().reset_index()

def pseudobulk_test(df):
    pb = pseudobulk(df)
    return stats.ttest_ind(np.log(pb.loc[pb.condition == "control", "count"]),
                           np.log(pb.loc[pb.condition == "treated", "count"])).pvalue

def false_positive_rates(n_experiments=300):
    p_cells, p_pb = [], []
    for _ in range(n_experiments):
        df = simulate_experiment()
        p_cells.append(cell_level_test(df))
        p_pb.append(pseudobulk_test(df))
    return pd.Series({"cells as n": np.mean(np.array(p_cells) < 0.05),
                      "organoids as n (pseudobulk)": np.mean(np.array(p_pb) < 0.05)},
                     name="false positive rate (nominal 5%)")

false_positive_rates()

### 6d — A marker where its cells are not (Visium HD)

**Question:** *Ppp1r1b* (DARPP-32) marks the medium spiny neurons of the striatum, on the left of the crop.
How far into the white matter and cortex does its signal reach? Compare with the oligodendrocyte marker
*Mbp*, which marks the white matter.

```
Lateral spillover
├── Aggregate      → 16 µm bins
├── Profile        → for each column of bins (left → right), mean normalised expression over all rows
├── Plot           → profiles of Ppp1r1b and Mbp vs distance (µm), log scale
└── Read           → where does Ppp1r1b fall to its cortical background level?
```

*Discussion question:* part of the signal outside the striatum is real (some cortical neurons express
*Ppp1r1b*), part is mRNA that diffused laterally during permeabilisation, or ambient RNA. How would you
tell the two apart? What does this mean for the "single-cell resolution" of a 2 µm grid?

In [ ]:
def column_profile(counts, bins, gene):
    expr = bin_score(counts, bins, [gene])
    return pd.Series(expr).groupby(bins["col"].to_numpy()).mean()

def plot_profiles(genes=("Ppp1r1b", "Mbp", "Slc17a7"), size=16):
    agg, agg_bins = aggregate(vh_counts, vh_bins, size)
    plt.figure(figsize=(8, 3.5))
    for g in genes:
        profile = column_profile(agg, agg_bins, g)
        plt.plot(profile.index * size, profile.values, label=g)
    plt.yscale("log")
    plt.xlabel("distance from the left edge of the crop (µm)  —  striatum → white matter → cortex → pia")
    plt.ylabel("mean log1p(1000 × count / total)")
    plt.legend()
    plt.show()

plot_profiles()

## Wrap-up

| | Bulk / qPCR | Single cell | Visium HD bin |
|---|---|---|---|
| molecules per observation | 10⁷ | 10³–10⁴ | 10–10³ |
| genes read with CV < 30% | almost all | a few hundred | almost none |
| a single marker gene | reliable | weak | unreliable |
| what to do | — | signatures, clusters, pseudobulk | signatures, bigger bins, smoothing, single-cell reference |

**References.**
Svensson (2020) *Droplet scRNA-seq is not zero-inflated*, Nat Biotech ·
Neufeld et al. (2024) *Inference after latent variable estimation for single-cell RNA sequencing data*, Biostatistics ·
Squair et al. (2021) *Confronting false discoveries in single-cell differential expression*, Nat Commun ·
Gao, Bien & Witten (2024) *Selective inference for hierarchical clustering*, JASA ·
Oliveira et al. (2025) *High-definition spatial transcriptomic profiling of immune cell populations in colorectal cancer* (Visium HD), Nat Genet ·
Cable et al. (2022) *Robust decomposition of cell type mixtures in spatial transcriptomics* (RCTD), Nat Biotech ·
Tasic et al. (2018) *Shared and distinct transcriptomic cell types across neocortical areas*, Nature.